# Notebook 4: the dead zone, and a TDI feature split

Two independent parts. Run A, then B, or either alone.

**Part A, regression.** ST-RAE forgives any prediction inside the credible band, so the
quantity a model should chase is the nearest bound, not the point estimate. Refitting by
majorise-minimise against `clip(p, lo, hi)` is worth, on a fast stand-in tested locally:

| | iteration 0 | iteration 1 | iteration 2 |
| :--- | ---: | ---: | ---: |
| macro ST-RAE | 0.7055 | **0.6473** | 0.6541 |
| macro Spearman | 0.6090 | **0.6381** | 0.6282 |

One step, +0.0291 of Spearman, sign consistent 4 of 4. The no-PCA change that took us from
rank 78 to 54 was +0.0125. A second step regresses, so one is the sweet spot.

The caveat this notebook exists to settle: that stand-in starts at 0.6090, below our real
blend at 0.6306, so part of the gain may be a weak model catching up rather than a strong
one improving. Here it runs on the actual arms.

Also folded in: an instrument arm, isotonic from the single-concentration screen onto
pIC50. The screen alone correlates -0.52, -0.65, -0.37, -0.75 with the labels, it covers
the blind set at 100%, and as an arm it measured +0.0037, 0.0000, +0.0092, +0.0007
locally, concentrated on CYP2D6 where it is least correlated with our models.

**Part B, TDI.** TabPFN lost on that track. What notebook 3 did show is that the two
endpoints want different features: CYP2D6 reached 0.6169 out-of-fold AUC on embeddings
against 0.5896 on descriptors, while CYP3A4 went the other way, 0.7766 against 0.8186.
Part B builds both feature sets for both endpoints and blends them.

In [ ]:
import subprocess
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout or 'NO GPU')

In [ ]:
PAYLOAD_FILE = '/content/drive/MyDrive/cyp_payloads/mini3_d3108f9c6e.tgz'

import os, tarfile
os.makedirs('/content/cache', exist_ok=True)
from google.colab import drive
drive.mount('/content/drive')
with tarfile.open(PAYLOAD_FILE) as t:
    t.extractall('/content/cache')

# Results go to Drive so a runtime disconnect cannot take them.
OUTDIR = '/content/results'
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTDIR = '/content/drive/MyDrive/cyp_results'
except Exception as e:
    print('Drive unavailable:', type(e).__name__, '- download early and often')
os.makedirs(OUTDIR, exist_ok=True)
print('writing to', OUTDIR)

In [ ]:
!pip -q install tabpfn==2.2.1 lightgbm rdkit

In [ ]:
import gc, json, time, numpy as np, pandas as pd, torch
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

CACHE = Path('/content/cache'); OUT = Path(OUTDIR)
ISO = ['CYP1A2', 'CYP2C9', 'CYP2D6', 'CYP3A4']
N_FOLDS = 5
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

meta = pd.read_csv(CACHE / 'meta_train.csv.gz'); mete = pd.read_csv(CACHE / 'meta_test.csv.gz')
hold = np.load(CACHE / 'holdout_mask.npy')
G = np.load(CACHE / 'scaffold_groups.npy', allow_pickle=True)
EMB = np.load(CACHE / 'emb_medium_inhib.npy').astype(np.float32)
EMB_TE = np.load(CACHE / 'emb_medium_test.npy').astype(np.float32)
QH, QH_TE = np.load(CACHE / 'qhts_tr.npy'), np.load(CACHE / 'qhts_te.npy')
AUX, AUX_TE = np.load(CACHE / 'aux_train.npy'), np.load(CACHE / 'aux_test.npy')
SM_TR = meta['SMILES'].to_numpy(); SM_TE = mete['SMILES'].to_numpy()

soft = lambda p, lo, hi: np.clip(p - hi, 0, None) + np.clip(lo - p, 0, None)
st_rae = lambda y, p, lo, hi: float(soft(p, lo, hi).sum()
                                    / soft(np.full_like(y, y.mean()), lo, hi).sum())

def bands(iso):
    c = f'{iso}_pIC50_direct_inhibition'
    y = meta[c].to_numpy(float)
    lo = meta[f'{c}_conf_low'].to_numpy(float); hi = meta[f'{c}_conf_high'].to_numpy(float)
    ok = np.isfinite(y) & np.isfinite(lo) & np.isfinite(hi)
    return y, lo, hi, np.flatnonzero(ok & ~hold), np.flatnonzero(ok & hold)

RESULTS = []
def record(**kw):
    RESULTS.append(kw); json.dump(RESULTS, open(OUT / 'results_nb4.json', 'w'), indent=2)

print('device', DEVICE, '|', {i: len(bands(i)[3]) for i in ISO})

## Part A, the dead zone on the real arms

For each arm: fit normally, take the out-of-fold predictions, clip them into each
compound's credible band, refit against those clipped values. Using out-of-fold
predictions for the clipping is what stops the target collapsing onto the model's own
training fit.

In [ ]:
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors
RDLogger.DisableLog('rdApp.*')
names = [d[0] for d in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(names)
gen = AllChem.GetMorganGenerator(radius=2, fpSize=2048)

def featurise(sm):
    X = np.zeros((len(sm), 2048 + len(names)), np.float32)
    for i, s in enumerate(sm):
        m = Chem.MolFromSmiles(s) if isinstance(s, str) else None
        if m is None:
            continue
        X[i, :2048] = gen.GetFingerprintAsNumPy(m)
        X[i, 2048:] = np.nan_to_num(np.asarray(calc.CalcDescriptors(m), np.float32),
                                    nan=0.0, posinf=0.0, neginf=0.0)
    return X

t0 = time.time()
DESC, DESC_TE = featurise(SM_TR), featurise(SM_TE)
print('descriptors', DESC.shape, f'({time.time()-t0:.0f}s)')

TREE_X = np.nan_to_num(np.hstack([DESC, QH, AUX]).astype(np.float32))
TREE_TE = np.nan_to_num(np.hstack([DESC_TE, QH_TE, AUX_TE]).astype(np.float32))
TAB_X = np.nan_to_num(np.hstack([EMB, QH, AUX]).astype(np.float32))
TAB_TE = np.nan_to_num(np.hstack([EMB_TE, QH_TE, AUX_TE]).astype(np.float32))

In [ ]:
import lightgbm as lgb
from tabpfn import TabPFNRegressor
KW = dict(device=DEVICE, random_state=42, ignore_pretraining_limits=True)
LP = dict(objective='regression', learning_rate=0.03, num_leaves=31, min_data_in_leaf=20,
          feature_fraction=0.3, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
          verbosity=-1, num_threads=4)

FIT = {'trees':  (TREE_X, TREE_TE, lambda a, b, c: lgb.train(LP, lgb.Dataset(a, b), 700).predict(c)),
       'tabpfn': (TAB_X,  TAB_TE,  lambda a, b, c: TabPFNRegressor(**KW).fit(a, b).predict(c))}

for arm, (X, XT, fp) in FIT.items():
    for iso in ISO:
        y, lo, hi, dev, ho = bands(iso)
        folds = list(GroupKFold(N_FOLDS).split(dev, dev, G[dev]))
        yd, lod, hid = y[dev], lo[dev], hi[dev]
        target = yd.copy()
        t0 = time.time()
        for it in range(2):                       # iteration 0 plain, iteration 1 dead zone
            oof = np.zeros(len(dev))
            for tr, va in folds:
                oof[va] = fp(X[dev[tr]], target[tr], X[dev[va]])
            hp = fp(X[dev], target, X[ho])
            allr = np.concatenate([dev, ho])
            tgt_all = np.concatenate([target, y[ho]])
            tp = fp(X[allr], tgt_all, XT)
            np.savez_compressed(OUT / f'dz_{arm}_{iso}_it{it}.npz',
                                oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
            s, r = st_rae(yd, oof, lod, hid), float(spearmanr(yd, oof).statistic)
            record(part='A', arm=arm, iso=iso, iteration=it, dev_st_rae=s, dev_spearman=r,
                   holdout_st_rae=st_rae(y[ho], hp, lo[ho], hi[ho]),
                   holdout_spearman=float(spearmanr(y[ho], hp).statistic),
                   seconds=round(time.time() - t0, 1))
            print(f'  {arm:6s} {iso} it{it}: dev ST-RAE {s:.4f} rho {r:.4f} '
                  f'| holdout ST-RAE {RESULTS[-1]["holdout_st_rae"]:.4f} '
                  f'({time.time()-t0:.0f}s)', flush=True)
            target = np.clip(oof, lod, hid)       # the majorise-minimise step
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

In [ ]:
# The instrument arm: isotonic from the single-concentration screen onto pIC50, fitted
# out of fold so it is never scored on rows it saw. The screen covers the blind set.
for j, iso in enumerate(ISO):
    y, lo, hi, dev, ho = bands(iso)
    folds = list(GroupKFold(N_FOLDS).split(dev, dev, G[dev]))
    oof = np.full(len(dev), np.nan)
    for tr, va in folds:
        m = IsotonicRegression(increasing=False, out_of_bounds='clip')
        m.fit(np.nan_to_num(AUX[dev[tr], j]), y[dev[tr]])
        oof[va] = m.predict(np.nan_to_num(AUX[dev[va], j]))
    m = IsotonicRegression(increasing=False, out_of_bounds='clip')
    m.fit(np.nan_to_num(AUX[dev, j]), y[dev])
    hp = m.predict(np.nan_to_num(AUX[ho, j]))
    allr = np.concatenate([dev, ho])
    m2 = IsotonicRegression(increasing=False, out_of_bounds='clip')
    m2.fit(np.nan_to_num(AUX[allr, j]), y[allr])
    tp = m2.predict(np.nan_to_num(AUX_TE[:, j]))
    np.savez_compressed(OUT / f'instrument_{iso}.npz', oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
    r = float(spearmanr(y[dev], oof).statistic)
    record(part='A', arm='instrument', iso=iso, iteration=0, dev_spearman=r,
           dev_st_rae=st_rae(y[dev], oof, lo[dev], hi[dev]),
           holdout_spearman=float(spearmanr(y[ho], hp).statistic),
           holdout_st_rae=st_rae(y[ho], hp, lo[ho], hi[ho]), seconds=0)
    print(f'  instrument {iso}: dev rho {r:.4f}', flush=True)

r = pd.DataFrame([x for x in RESULTS if x['part'] == 'A'])
print()
print(r.pivot_table(index=['arm', 'iteration'], values=['dev_st_rae', 'dev_spearman',
                                                        'holdout_st_rae']).round(4).to_string())

## Part B, TDI with the right features per endpoint

In [ ]:
from sklearn.metrics import roc_auc_score, roc_curve
ENDP = ['CYP2D6', 'CYP3A4']
LPC = dict(LP); LPC['objective'] = 'binary'

def best_mcc(y, p):
    fpr, tpr, _ = roc_curve(y, p); pr = y.mean()
    tp, fn = tpr * pr, (1 - tpr) * pr
    fp, tn = fpr * (1 - pr), (1 - fpr) * (1 - pr)
    den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return float(np.nanmax(np.where(den > 0, (tp * tn - fp * fn) / np.maximum(den, 1e-12), 0)))

FEATS = {'desc': (TREE_X, TREE_TE), 'emb': (TAB_X, TAB_TE)}
store = {}
for ep in ENDP:
    yv = meta[f'{ep}_is_TDI'].to_numpy(float)
    ok = np.isfinite(yv)
    dev, ho = np.flatnonzero(ok & ~hold), np.flatnonzero(ok & hold)
    folds = list(GroupKFold(N_FOLDS).split(dev, dev, G[dev]))
    for fname, (X, XT) in FEATS.items():
        oof = np.zeros(len(dev))
        for tr, va in folds:
            oof[va] = lgb.train(LPC, lgb.Dataset(X[dev[tr]], yv[dev[tr]]), 700).predict(X[dev[va]])
        hp = lgb.train(LPC, lgb.Dataset(X[dev], yv[dev]), 700).predict(X[ho])
        allr = np.concatenate([dev, ho])
        tp = lgb.train(LPC, lgb.Dataset(X[allr], yv[allr]), 700).predict(XT)
        np.savez_compressed(OUT / f'tdi2_{fname}_{ep}.npz', oof=oof, dev=dev, hold=hp, ho=ho, test=tp)
        store[(ep, fname)] = (oof, hp, yv, dev, ho)
        record(part='B', arm=fname, iso=ep, oof_auc=roc_auc_score(yv[dev], oof),
               holdout_auc=roc_auc_score(yv[ho], hp), oof_best_mcc=best_mcc(yv[dev], oof))
        print(f'  {ep} {fname}: OOF AUC {RESULTS[-1]["oof_auc"]:.4f} | '
              f'holdout {RESULTS[-1]["holdout_auc"]:.4f} | best MCC {RESULTS[-1]["oof_best_mcc"]:.4f}',
              flush=True)
    a, _, _, _, _ = store[(ep, 'desc')]; b, _, _, _, _ = store[(ep, 'emb')]
    best = max(((roc_auc_score(yv[dev], (1 - w) * a + w * b), w) for w in np.linspace(0, 1, 21)))
    print(f'  {ep} blended: OOF AUC {best[0]:.4f} at emb weight {best[1]:.2f}', flush=True)
    record(part='B', arm='blend', iso=ep, oof_auc=best[0], emb_weight=best[1],
           oof_best_mcc=best_mcc(yv[dev], (1 - best[1]) * a + best[1] * b))

In [ ]:
import shutil
from google.colab import files
r = pd.DataFrame(RESULTS)
print(r.to_string(index=False))
shutil.make_archive('/content/notebook4_results', 'zip', OUT)
files.download('/content/notebook4_results.zip')